# Ön eğitim · 1M / 10M / 30M · Colab **ve** Kaggle
Aynı defter iki yerde çalışır. `ai/egitim/egit.py`'yi çağırır, bitince kayıp grafiğini ve birkaç örnek kart/hikâye gösterir.

**Önce (bir kez):** tokenizer altın setsiz veriyle yeniden eğitilip dondurulmuş, `filtre.py` ve `hazirla.py` çalıştırılmış olmalı. Üç dosya tek klasörde:
`train.bin`, `val.bin`, `tokenizer.json`

**Colab:** bu üç dosyayı Drive'da `kaydir_veri/egitim/` klasörüne koy → Çalışma zamanı → **T4 GPU** → 1. hücrede `BOYUT` seç → *Tümünü çalıştır*. Model `kaydir_veri/modeller/<boyut>/`'a kaydedilir; oturum koparsa aynı defteri yeniden çalıştır, **kaldığı yerden devam eder**.

**Kaggle:** üç dosyayı bir Kaggle *Dataset*'i olarak yükle (ör. adı `buyruk-egitim`) → defterde *Add Input* ile ekle → Ayarlar: **GPU T4 ×2**, **Internet on** → *Run All*. Çıktı `/kaggle/working/modeller/` (oturum sonunda *Output*'tan indir). Kaggle 12 saatte keser; `SURE_SAAT` bunun altında durdurup `son.pt` kaydeder. Devam etmek için önceki çıktıyı Dataset yapıp `ONCEKI`'ye yolunu yaz.

Ne kadar sürer (T4, kaba): 1M ~1 saat altı · 10M birkaç saat · 30M yarım gün ve üstü (veri miktarına göre değişir; ilk satırda adım sayısı yazar).

## 1 · Ayar

In [ ]:
BOYUT = '1m'          # '1m' · '10m' · '30m'  (ai/model/boyutlar/ altındaki yaml)
DAL = 'main'          # AI ve Docs depolarının dalı
SURE_SAAT = 11        # en fazla bu kadar saat eğit (Kaggle 12 saatte keser); 0 = sınırsız
ADIM = None           # None: yaml'daki `tur`dan hesaplanır (veri 4 kez okunur). Hızlı deneme için ör. 500
KAGGLE_VERI = ''      # Kaggle'da Dataset klasörü; boş bırakırsan /kaggle/input altında train.bin aranır
ONCEKI = ''           # Kaggle'da önceki oturumun son.pt'si (devam için), ör. '/kaggle/input/buyruk-1m-oturum1/son.pt'

## 2 · Hazırlık: depoları indir, veriyi bul

In [ ]:
import os, sys, glob, shutil, subprocess
KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))
if KAGGLE:
    if not KAGGLE_VERI:
        bulunan = glob.glob('/kaggle/input/**/train.bin', recursive=True)
        assert bulunan, "train.bin bulunamadı: Dataset'i 'Add Input' ile ekle"
        KAGGLE_VERI = os.path.dirname(bulunan[0])
    VERI = KAGGLE_VERI
    CIKTI = f'/kaggle/working/modeller/{BOYUT}'
else:
    from google.colab import drive; drive.mount('/content/drive')
    VERI = '/content/drive/MyDrive/kaydir_veri/egitim'
    CIKTI = f'/content/drive/MyDrive/kaydir_veri/modeller/{BOYUT}'
for depo in ('AI', 'Docs'):           # Docs: kart biçimi tema.json'u oradan okur
    yol = f'/tmp/{depo}'
    if os.path.isdir(f'{yol}/.git'):
        subprocess.run(['git', '-C', yol, 'fetch', '-q', '--depth', '1', 'origin', DAL], check=True)
        subprocess.run(['git', '-C', yol, 'checkout', '-q', '-B', DAL, 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', DAL, f'https://github.com/IleriProgramlamaProje/{depo}', yol], check=True)
TOKENIZER = f'{VERI}/tokenizer.json'
for d in ('train.bin', 'val.bin', 'tokenizer.json'):
    assert os.path.exists(f'{VERI}/{d}'), f'{VERI}/{d} yok'
os.makedirs(CIKTI, exist_ok=True)
if ONCEKI and not os.path.exists(f'{CIKTI}/son.pt'):
    shutil.copy(ONCEKI, f'{CIKTI}/son.pt')
    kayip = os.path.join(os.path.dirname(ONCEKI), 'kayip.csv')
    if os.path.exists(kayip): shutil.copy(kayip, f'{CIKTI}/kayip.csv')
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK (Çalışma zamanı → T4 GPU seç)')
print('veri', VERI, '· çıktı', CIKTI, '·', subprocess.run(['git', '-C', '/tmp/AI', 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip())

## 3 · Eğit
Her 250 adımda train/val kaybı yazar; `*en iyi` olan satırda `en_iyi.pt` güncellenir. Val 6 ölçüm iyileşmezse kendisi durur (ezber başlamış demektir).

In [ ]:
komut = [sys.executable, '/tmp/AI/ai/egitim/egit.py', '--boyut', f'/tmp/AI/ai/model/boyutlar/{BOYUT}.yaml',
         '--veri', VERI, '--tokenizer', TOKENIZER, '--cikti', CIKTI, '--sure-saat', str(SURE_SAAT)]
if ADIM: komut += ['--adim', str(ADIM)]
if os.path.exists(f'{CIKTI}/son.pt'): komut.append('--devam')
!{' '.join(komut)}

## 4 · Kayıp grafiği
İki çizgi birlikte iniyorsa iyi. Train inip val yükseliyorsa model ezberliyor: en iyi nokta `en_iyi.pt`'de saklı.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
k = pd.read_csv(f'{CIKTI}/kayip.csv')
plt.figure(figsize=(8, 4)); plt.plot(k.adim, k.train, label='train'); plt.plot(k.adim, k.val, label='val')
plt.xlabel('adım'); plt.ylabel('kayıp'); plt.legend(); plt.grid(alpha=.3); plt.show()
print(f"en iyi val {k.val.min():.3f} (adım {int(k.adim[k.val.idxmin()])}) · son val {k.val.iloc[-1]:.3f}")

## 5 · Örnekler
Kartta model durum satırı + konuşanı alır, gövdeyi kendisi yazar. Sonda kaç kartın biçimce geçerli olduğu yazılır (karar noktasında bakılacak sayı).

In [ ]:
!python /tmp/AI/ai/ornekleme/ornek_uret.py --model {CIKTI}/en_iyi.pt --tokenizer {TOKENIZER} --tur kart --adet 100 --sicaklik 0.8 --cikti {CIKTI}/ornek_kartlar.jsonl | tail -40

In [ ]:
!python /tmp/AI/ai/ornekleme/ornek_uret.py --model {CIKTI}/en_iyi.pt --tokenizer {TOKENIZER} --tur hikaye --adet 3 --baslangic "Bir gün"